# DaTikZ-V4 Quality Evaluation

  > Evaluates the quality of VLM-generated TikZ figure descriptions against the actual rendered images using Mistral Small 2506 as a strict judge. The scored dataset is pushed to Hugging Face for downstream filtering of training-quality samples.

## Pipeline

┌─────────────────┐
│  DaTikZ-V4 HF   │
│  (first 10k)    │
└────────┬────────┘
         │ png_image + vlm_description
         ▼
┌─────────────────┐     ┌──────────────┐
│  Async Worker   │────▶│ Token Bucket │
│  Pool (20 ctx)  │     │  (5 req/s)   │
└────────┬────────┘     └──────┬───────┘
         │                     │
         │    Mistral Small 2506 (vision)
         │    ┌─────────────────────────┐
         │    │ score, accuracy,        │
         │    │ completeness, clarity,  │
         │    │ geometric_precision,    │
         │    │ training_usable,        │
         │    │ needs_recaption,        │
         │    │ error_types, notes      │
         │    └────────────┬────────────┘
         ▼                 │
┌─────────────────┐        │
│  results.jsonl  │◄───────┘
│  (append-safe)  │
└────────┬────────┘
         │ merge with original dataset
         ▼
┌─────────────────────────┐
│  HF Dataset (scored)    │
│  with all eval metrics  │
└─────────────────────────┘

## Config

| Parameter | Value |
|-----------|-------|
| Model | Mistral Small 2506 |
| Rate limit | 5 req/s |
| Workers | 20 concurrent |
| Samples | 10,000 |
| Est. runtime | ~35-40 min |

## Output

Scored dataset pushed to Hugging Face with these added columns:

| Column | Description |
|--------|-------------|
| `score` | Overall quality (1-10) |
| `accuracy` | Visual fact correctness |
| `completeness` | All elements mentioned |
| `clarity` | Unambiguous & easy to follow |
| `geometric_precision` | Positions, shapes, curves, arrows correct |
| `training_usable` | Score ≥ 8.5 and no major errors |
| `needs_recaption` | Should be re-described |
| `error_types` | List of error categories |
| `notes` | Strict reasoning from evaluator |
| `eval_success` | Whether the API call succeeded |



In [ ]:
!pip install datasets httpx pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 134.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import asyncio
import json
import time
import base64
from io import BytesIO
from pathlib import Path

import httpx
from datasets import load_dataset
from PIL import Image
from tqdm.notebook import tqdm

MISTRAL_API_KEY = ""
MODEL = "mistral-small-2506"
API_URL = "https://api.mistral.ai/v1/chat/completions"
RATE_LIMIT = 5          # req/sec
CONCURRENT_WORKERS = 20
TOTAL_SAMPLES = 20000
RESULTS_FILE = "results.jsonl"
PROGRESS_FILE = "progress.json"

SEM = asyncio.Semaphore(CONCURRENT_WORKERS)
rate_bucket = RATE_LIMIT
last_refill = time.monotonic()
bucket_lock = asyncio.Lock()

In [3]:
async def acquire_rate():
    global rate_bucket, last_refill
    async with bucket_lock:
        now = time.monotonic()
        elapsed = now - last_refill
        rate_bucket = min(RATE_LIMIT, rate_bucket + elapsed * RATE_LIMIT)
        last_refill = now
        if rate_bucket < 1:
            wait = (1 - rate_bucket) / RATE_LIMIT
            await asyncio.sleep(wait)
            rate_bucket = 1
            last_refill = time.monotonic()
        rate_bucket -= 1

In [4]:
SYSTEM_PROMPT = """You are a strict quality evaluator for TikZ figure descriptions.

Your task is to judge whether the description is good enough to be used as training data for a text-to-TikZ generation model.

Be strict. A description should score high only if it can be used to recreate the image accurately.

Evaluate:
- accuracy: whether all stated visual facts are correct.
- completeness: whether all important visible elements are mentioned.
- clarity: whether the description is unambiguous and easy to follow.
- geometric_precision: whether positions, directions, shapes, sizes, connections, symmetry, line thickness, arrowheads, curves, and labels are described correctly.

Important scoring rules:
- Wrong object count is a major error.
- Wrong shape type is a major error.
- Wrong arrow direction is a major error.
- Saying curved when the line/arrow is straight is a major error.
- Saying straight when the line/arrow is curved is a major error.
- Confusing dots, arrowheads, endpoints, or nodes is a major error.
- Wrong connection point, such as corner vs edge vs center, is a major error.
- Wrong relative position, such as left/right/top/bottom/inside/outside, is a major error.
- Wrong line thickness, fill, color, or label presence should reduce the score.
- Hallucinated elements should reduce the score heavily.
- Missing small details is acceptable only if the main geometry is fully correct.
- A caption with correct main idea but wrong geometry should usually score 5-7, not 8-10.
- A caption suitable for training should usually score at least 8.5.

Return ONLY valid JSON. No markdown, no extra text.

Use this JSON schema:
{
  "score": <1-10>,
  "accuracy": <1-10>,
  "completeness": <1-10>,
  "clarity": <1-10>,
  "geometric_precision": <1-10>,
  "training_usable": true/false,
  "needs_recaption": true/false,
  "error_types": [
    "wrong_object_count",
    "wrong_shape",
    "wrong_position",
    "wrong_connection",
    "wrong_arrow_direction",
    "wrong_curve_or_line_type",
    "wrong_line_style",
    "wrong_color_or_fill",
    "wrong_label",
    "hallucinated_element",
    "missing_element",
    "ambiguous_description"
  ],
  "notes": "<brief strict reasoning>"
}
"""

In [5]:


def encode_image(pil_image):
    buf = BytesIO()
    pil_image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def build_payload(png_image, vlm_description):
    b64 = encode_image(png_image)
    return {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text", "text": f"Rate this description against the TikZ figure:\n\n{vlm_description}"},
                {"type": "image_url", "image_url": f"data:image/png;base64,{b64}"}
            ]}
        ],
        "temperature": 0,
        "max_tokens": 800
    }

In [6]:
EXPECTED_FIELDS = ["score", "accuracy", "completeness", "clarity", "geometric_precision",
                   "training_usable", "needs_recaption", "error_types", "notes"]

async def evaluate_one(client, idx, png_image, vlm_desc, pbar):
    async with SEM:
        await acquire_rate()
        for attempt in range(3):
            try:
                resp = await client.post(
                    API_URL,
                    headers={"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"},
                    json=build_payload(png_image, vlm_desc),
                    timeout=httpx.Timeout(60)
                )
                if resp.status_code == 200:
                    content = resp.json()["choices"][0]["message"]["content"]
                    result = json.loads(content)
                    result["_idx"] = idx
                    result["_success"] = True
                    result["_raw_response"] = content
                    return result
                elif resp.status_code == 429:
                    wait = float(resp.headers.get("retry-after", 5))
                    await asyncio.sleep(wait * (attempt + 1))
                else:
                    await asyncio.sleep(2 ** attempt)
            except json.JSONDecodeError:
                await asyncio.sleep(2 ** attempt)
            except Exception:
                await asyncio.sleep(2 ** attempt)
        pbar.update(1)
        return {"_idx": idx, "_success": False, "score": None, "error_types": [], "notes": "failed after 3 retries"}

In [ ]:
async def main():
    print("Loading dataset...")
    ds = load_dataset("nllg/DaTikZ-V4", split="train", streaming=True).take(TOTAL_SAMPLES)

    # resume
    done = set()
    if Path(PROGRESS_FILE).exists():
        done = set(json.loads(open(PROGRESS_FILE).read()))
        print(f"Resuming — {len(done)} already done")

    pbar = tqdm(total=TOTAL_SAMPLES, initial=len(done))
    lock = asyncio.Lock()

    # lazily collect samples to avoid double-iteration on resume
    samples = [(i, s) for i, s in enumerate(ds)]

    async def process(idx, sample):
        if idx in done:
            pbar.update(1)
            return
        result = await evaluate_one(client, idx, sample["png_image"], sample["vlm_description"], pbar)
        async with lock:
            with open(RESULTS_FILE, "a") as f:
                f.write(json.dumps(result) + "\n")
            done.add(idx)
            with open(PROGRESS_FILE, "w") as f:
                json.dump(list(done), f)
        pbar.update(1)

    async with httpx.AsyncClient() as client:
        chunk_size = 500
        for start in range(0, len(samples), chunk_size):
            chunk = samples[start:start + chunk_size]
            await asyncio.gather(*[process(i, s) for i, s in chunk])

    pbar.close()
    print("Done.")

await main()

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/86 [00:00<?, ?it/s]

  0%|          | 0/20000 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

rows = [json.loads(l) for l in open(RESULTS_FILE) if l.strip()]
df = pd.DataFrame(rows)
success = df[df["_success"] == True]

print(f"Total: {len(df)} | Success: {len(success)} | Failed: {len(df) - len(success)}\n")

for col in ["score", "accuracy", "completeness", "clarity", "geometric_precision"]:
    vals = success[col].dropna()
    print(f"{col:25s}: mean={vals.mean():.2f}  median={vals.median():.1f}  min={vals.min():.0f}  max={vals.max():.0f}")

print(f"\ntraining_usable: {success['training_usable'].value_counts().to_dict()}")
print(f"needs_recaption: {success['needs_recaption'].value_counts().to_dict()}")
print(f"\nScore histogram:")
success["score"].value_counts().sort_index().plot(kind="bar", title="Score Distribution")

In [ ]:
import pandas as pd

rows = [json.loads(l) for l in open(RESULTS_FILE) if l.strip()]
df = pd.DataFrame(rows)
success = df[df["_success"] == True]

print(f"Total: {len(df)} | Success: {len(success)} | Failed: {len(df) - len(success)}\n")

for col in ["score", "accuracy", "completeness", "clarity", "geometric_precision"]:
    vals = success[col].dropna()
    print(f"{col:25s}: mean={vals.mean():.2f}  median={vals.median():.1f}  min={vals.min():.0f}  max={vals.max():.0f}")

print(f"\ntraining_usable: {success['training_usable'].value_counts().to_dict()}")
print(f"needs_recaption: {success['needs_recaption'].value_counts().to_dict()}")
print(f"\nScore histogram:")
success["score"].value_counts().sort_index().plot(kind="bar", title="Score Distribution")

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Reload original data and merge with scores
original_ds = load_dataset("nllg/DaTikZ-V4", split="train", streaming=True).take(TOTAL_SAMPLES)

# Index results by _idx for fast lookup
results_by_idx = {}
for line in open(RESULTS_FILE):
    r = json.loads(line)
    results_by_idx[r["_idx"]] = r

# Build output rows
output_rows = []
for i, sample in enumerate(original_ds):
    r = results_by_idx.get(i, {})
    output_rows.append({
        "png_image": sample["png_image"],
        "vlm_description": sample["vlm_description"],
        "score": r.get("score"),
        "accuracy": r.get("accuracy"),
        "completeness": r.get("completeness"),
        "clarity": r.get("clarity"),
        "geometric_precision": r.get("geometric_precision"),
        "training_usable": r.get("training_usable"),
        "needs_recaption": r.get("needs_recaption"),
        "error_types": json.dumps(r.get("error_types", [])),
        "notes": r.get("notes", ""),
        "eval_success": r.get("_success", False),
    })

output_ds = Dataset.from_list(output_rows)
output_ds.push_to_hub(f"{HF_USERNAME}/{HF_DATASET_NAME}", private=False)
print(f"Pushed to https://huggingface.co/datasets/{HF_USERNAME}/{HF_DATASET_NAME}")

In [ ]:
verify = load_dataset(f"{HF_USERNAME}/{HF_DATASET_NAME}", split="train")
print(f"Rows: {len(verify)}")
print(f"Columns: {verify.column_names}")
print(f"Mean score: {sum(s for s in verify['score'] if s) / sum(1 for s in verify['score'] if s):.2f}")
